# CAN Bus Security and Intrusion Detection

**Module 05 · Cybersecurity (ISO/SAE 21434)** · Notebook 4 of 4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/milinpatel07/Autonomous-Driving_AI-Safety-and-Security/blob/main/05_Cybersecurity_ISO21434/04_can_bus_intrusion_detection.ipynb)

| Level | Time | Prerequisites | Standards |
|---|---|---|---|
| Advanced | ~75 min | [TARA](02_tara.ipynb), [Attack surface analysis](03_attack_surface_analysis.ipynb) | ISO/SAE 21434:2021, UN R155, ISO 11898-1:2024, AUTOSAR SecOC |

**Author:** Milin Patel · Hochschule Kempten · MIT Licence

## Learning objectives

After working through this notebook you will be able to:

1. Explain the CAN 2.0 frame format, bitwise arbitration and the three properties that make the bus hard to defend: **broadcast**, **priority by identifier** and **no sender authentication**.
2. Generate realistic synthetic CAN traffic (per-ECU clocks, jitter, event-driven frames, counters and checksums) so that a *known-ground-truth* stream is available for testing detectors.
3. Implement three families of intrusion detection from first principles — **timing/frequency**, **payload/semantic plausibility** and **unsupervised ML** (Isolation Forest, identifier entropy) — plus a clock-skew fingerprint in the spirit of CIDS.
4. Evaluate detectors the way a vehicle programme must: precision/recall/F1, ROC, **detection latency** and **false alarms per hour of driving**.
5. Quantify the cost of prevention with AUTOSAR SecOC (truncated MAC, freshness value) on classic CAN versus CAN FD, and relate the analysis to an ISO/SAE 21434 TARA and the UN R155 Annex 5 threat list.

> **Scope and intent.** This is defensive teaching material. The synthetic anomalies below exist only so that detectors can be *measured*; they reproduce, at a high level, patterns already documented in the open literature and public datasets. Nothing here targets a real vehicle, and no step requires a vehicle or a network.

## 1. Setup

Only NumPy, pandas, Matplotlib and scikit-learn are needed; everything runs on a CPU in well under two minutes.
All CAN traffic in this notebook is **simulated**.

> **Illustrative data:** values below are synthetic, chosen for teaching; they are not measurements.

In [ ]:
import heapq
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_curve, roc_auc_score, precision_recall_fscore_support

SEED = 42
rng = np.random.default_rng(SEED)
seeds = np.random.SeedSequence(SEED).spawn(8)          # independent streams per experiment part
OKABE_ITO = ["#E69F00", "#56B4E9", "#009E73", "#F0E442", "#0072B2", "#D55E00", "#CC79A7", "#000000"]
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

BITRATE = 500_000      # bit/s, a common classic-CAN rate for chassis/powertrain buses
print("Ready. Bit rate:", BITRATE // 1000, "kbit/s")

## 2. CAN in one page: frame, arbitration and why it is hard to secure

Classic CAN (CAN 2.0A, now specified in ISO 11898-1:2024) is a **multi-master broadcast bus**. A base data frame with an
11-bit identifier contains:

| Field | Bits | Purpose |
|---|---|---|
| SOF | 1 | start of frame (dominant) |
| Identifier | 11 | **message identity and priority** — not a sender address |
| RTR, IDE, r0 | 3 | frame-type flags |
| DLC | 4 | data length code (0–8 bytes) |
| Data | 0–64 | payload (signals packed by the OEM's communication matrix) |
| CRC + delimiter | 16 | 15-bit CRC protects against *random* errors only |
| ACK slot + delimiter | 2 | any receiver acknowledges |
| EOF + IFS | 7 + 3 | end of frame, inter-frame space |

Before bit stuffing a frame with $s$ data bytes has $47 + 8s$ bits. Because the controller inserts a stuff bit after
five equal bits, the **worst-case** length used in schedulability analysis (Davis et al., 2007) is

$$C(s) = \left(g + 8s + 13 + \left\lfloor \frac{g + 8s - 1}{4} \right\rfloor\right)\tau_{\mathrm{bit}}, \qquad g = 34 \text{ for 11-bit identifiers.}$$

**Arbitration.** Nodes that wish to send start together and transmit their identifier bit by bit. The bus is a
wired-AND: a dominant `0` overwrites a recessive `1`. A node that sends `1` but reads back `0` has lost and backs off.
Hence **the lowest identifier wins**.

Three properties follow, and they are the root cause of every CAN threat:

1. **Broadcast** — every node sees every frame, so traffic can be observed passively.
2. **Priority by identifier** — a node that keeps sending low identifiers can starve the bus.
3. **No sender authentication** — any node may emit any identifier; a receiver cannot tell which ECU sent a frame.

In [ ]:
def can_frame_bits(dlc, worst_case=True):
    """Bits on the wire for a classic CAN base frame (11-bit ID) incl. 3-bit IFS (Davis et al., 2007)."""
    g = 34
    bits = g + 8 * dlc + 13
    if worst_case:
        bits += (g + 8 * dlc - 1) // 4
    return bits

for dlc in (0, 4, 8):
    nom, wc = can_frame_bits(dlc, False), can_frame_bits(dlc, True)
    print(f"DLC={dlc}: {nom:3d} bits nominal, {wc:3d} bits worst case "
          f"-> {wc / BITRATE * 1e3:.3f} ms at {BITRATE // 1000} kbit/s")
FRAME_TIME = can_frame_bits(8) / BITRATE   # every frame in our simulation carries 8 data bytes

In [ ]:
def arbitrate(ids, width=11):
    """Bitwise CAN arbitration. Returns bit matrix, bus level per bit, bit index where each node lost, winner."""
    bits = np.array([[(i >> (width - 1 - b)) & 1 for b in range(width)] for i in ids])
    active = np.ones(len(ids), bool)
    lost_at = np.full(len(ids), -1)
    bus = []
    for b in range(width):
        level = bits[active, b].min()                  # wired-AND: dominant 0 wins
        bus.append(level)
        losers = active & (bits[:, b] != level)
        lost_at[losers] = b
        active &= ~losers
    return bits, np.array(bus), lost_at, int(np.flatnonzero(active)[0])

contenders = {"Steering (0x120)": 0x120, "ESC speed (0x0B0)": 0x0B0, "ESC brake (0x0A0)": 0x0A0, "Spurious (0x000)": 0x000}
bits, bus, lost_at, winner = arbitrate(list(contenders.values()))

grid = np.vstack([bits, bus]).astype(float)
for r, b in enumerate(lost_at):
    if b >= 0:
        grid[r, b + 1:] = np.nan                        # node stops driving the bus after losing
fig, ax = plt.subplots(figsize=(8, 2.8))
ax.imshow(np.ma.masked_invalid(grid), cmap=ListedColormap([OKABE_ITO[4], OKABE_ITO[1]]), aspect="auto")
for (r, c), v in np.ndenumerate(grid):
    ax.text(c, r, "-" if np.isnan(v) else int(v), ha="center", va="center", color="white" if v == 0 else "black")
ax.set_yticks(range(len(contenders) + 1), list(contenders) + ["Bus level"])
ax.set_xticks(range(11), [f"ID{10 - b}" for b in range(11)])
ax.set_xlabel("Identifier bit (most significant first)")
ax.set_title("CAN arbitration: dark = dominant 0, light = recessive 1, '-' = node has backed off")
ax.grid(False)
plt.tight_layout(); plt.show()
for n, b in zip(contenders, lost_at):
    print(f"{n:20s} " + ("WINS arbitration" if b < 0 else f"loses at identifier bit {10 - b}"))
print("\nThe lowest identifier always wins the bus: a node emitting 0x000 would dominate it.")

## 3. A realistic synthetic CAN traffic generator

Public CAN-IDS datasets exist — notably the HCRL *Car-Hacking* dataset (Seo et al., 2018; Song et al., 2020) and the
ROAD dataset of Verma et al. (2024), which also documents the pitfalls of the older ones — but we build our own so that
every property is known and the notebook stays offline. The generator follows a small, invented **communication matrix**
(the OEM's "DBC" in practice):

* **Per-ECU clocks.** Each ECU's oscillator deviates from nominal by a few tens of ppm (its *clock skew*). All identifiers sent by one ECU share that skew, but each identifier has its **own phase**. A common bug in teaching generators is a single global clock, which aligns all periodic identifiers perfectly and makes timing detection unrealistically easy; we avoid it.
* **Jitter** from task scheduling, plus queuing delay produced by an explicit **arbitration-based bus simulation** (lower identifier wins; a newer instance of the same frame overwrites an unsent older one in the transmit buffer).
* **Event-driven identifiers** (door lock, indicator stalk, HMI button) sent as a short burst of three repetitions.
* **Payload semantics.** Signals come from a simple vehicle-dynamics model (speed, steering, brake, RPM, gear). Protected frames carry a 4-bit **alive counter** and an 8-bit **checksum**, in the spirit of AUTOSAR E2E protection (simplified here to an inverted byte sum that also folds in the identifier).

In [ ]:
ECU_SKEW_PPM = {"ESC": 35.0, "EPS": -20.0, "ECM": 60.0, "TCU": -45.0, "BCM": 15.0, "HU": -80.0}

can_matrix = pd.DataFrame([
    # can_id, name,            sender, period_ms, event_rate_hz, jitter_ms
    (0x0A0, "BrakeStatus",      "ESC",   10,    np.nan, 0.05),
    (0x0B0, "VehicleSpeed",     "ESC",   10,    np.nan, 0.05),
    (0x120, "SteeringAngle",    "EPS",   10,    np.nan, 0.05),
    (0x1A0, "EngineRPM",        "ECM",   20,    np.nan, 0.08),
    (0x200, "AccelPedal",       "ECM",   20,    np.nan, 0.08),
    (0x2C0, "GearPosition",     "TCU",   50,    np.nan, 0.10),
    (0x316, "BodyStatus",       "BCM",  100,    np.nan, 0.20),
    (0x3E0, "ClimateStatus",    "HU",   200,    np.nan, 0.30),
    (0x440, "Odometer",         "ESC", 1000,    np.nan, 0.30),
    (0x5A0, "DoorLockEvent",    "BCM", np.nan,  0.02,   0.0),
    (0x5B0, "IndicatorEvent",   "BCM", np.nan,  0.10,   0.0),
    (0x6F0, "HMIButtonEvent",   "HU",  np.nan,  0.30,   0.0),
], columns=["can_id", "name", "sender", "period_ms", "event_rate_hz", "jitter_ms"])
ID_NAME = dict(zip(can_matrix.can_id, can_matrix.name))

# signal layout: can_id -> (signal, scale, offset, n_bytes);  raw = round(phys*scale + offset), big-endian from byte 0
SIGNALS = {
    0x0A0: ("brake_bar", 10, 0, 2),     0x0B0: ("speed_kmh", 100, 0, 2),
    0x120: ("steer_deg", 10, 32768, 2), 0x1A0: ("rpm", 4, 0, 2),
    0x200: ("pedal_pct", 2, 0, 1),      0x2C0: ("gear", 1, 0, 1),
    0x316: ("body_status", 1, 0, 1),    0x3E0: ("cabin_temp_c", 10, 0, 2),
    0x440: ("odo_km", 10, 0, 4),
}

def checksum(first7, can_id):
    """Simplified E2E-style checksum: inverted (byte sum + identifier bytes) mod 256."""
    return (((first7.astype(np.int64).sum(axis=1) + (can_id & 0xFF) + (can_id >> 8)) & 0xFF) ^ 0xFF).astype(np.uint8)

def encode(can_id, phys, counter):
    sig, scale, off, nb = SIGNALS[can_id]
    raw = np.clip(np.round(np.asarray(phys) * scale + off), 0, 2 ** (8 * nb) - 1).astype(np.int64)
    pl = np.zeros((raw.size, 8), np.uint8)
    for k in range(nb):
        pl[:, k] = (raw >> (8 * (nb - 1 - k))) & 0xFF
    pl[:, 6] = np.asarray(counter) % 16
    pl[:, 7] = checksum(pl[:, :7], can_id)
    return pl

def decode(can_id, pl):
    sig, scale, off, nb = SIGNALS[can_id]
    raw = np.zeros(len(pl), np.int64)
    for k in range(nb):
        raw = (raw << 8) | pl[:, k].astype(np.int64)
    return (raw - off) / scale

print(can_matrix.assign(can_id=can_matrix.can_id.map(hex)).to_string(index=False))

In [ ]:
GEAR_RATIOS, FINAL_DRIVE, WHEEL_R = np.array([3.6, 2.1, 1.4, 1.0, 0.8, 0.65]), 3.9, 0.32

def driving_profile(duration_s, rng, dt=0.01):
    """Simple longitudinal/lateral driving model sampled every dt seconds."""
    n = int(round(duration_s / dt)) + 1
    t = np.arange(n) * dt
    noise = rng.normal(size=(n, 2)) * np.sqrt(dt)
    v = np.empty(n); a = np.empty(n); steer = np.empty(n)
    vi = rng.uniform(5, 20); target = vi; next_change = 0.0; a_n = 0.0; s = 0.0; s_rate = 0.0
    for i in range(n):
        if t[i] >= next_change:
            target = rng.choice([0.0, 8.0, 14.0, 22.0, 30.0]); next_change = t[i] + rng.uniform(15, 40)
        a_n += -a_n * dt / 2.0 + 0.4 * noise[i, 0]
        acc = min(max(0.35 * (target - vi) + a_n, -4.0), 2.5)
        if vi <= 0.0 and acc < 0.0:
            acc = 0.0
        vi = max(vi + acc * dt, 0.0)
        s_rate += -s_rate * dt / 0.5 + 60.0 * noise[i, 1]
        s += s_rate * dt - s * dt / 5.0
        v[i], a[i], steer[i] = vi, acc, s
    gear = 1 + np.searchsorted([4, 8, 13, 18, 24], v)
    return {
        "t": t, "dt": dt, "speed_kmh": v * 3.6, "accel": a, "steer_deg": steer / (1 + v / 8),
        "brake_bar": np.where(a < -0.3, -a * 25.0, 0.0),
        "pedal_pct": np.clip(np.where(a > 0, 10 + 25 * a, 0.0), 0, 100), "gear": gear.astype(float),
        "rpm": np.maximum(800, v * 60 / (2 * np.pi * WHEEL_R) * GEAR_RATIOS[gear - 1] * FINAL_DRIVE),
        "body_status": np.full(n, 5.0), "cabin_temp_c": 21.5 + 0.0 * t,
        "odo_km": 12345.6 + np.cumsum(v) * dt / 1000,
    }

def frames_df(rel, can_id, src, label, pl):
    rel = np.maximum(np.atleast_1d(np.asarray(rel, float)), 0.0)
    df = pd.DataFrame({"rel": rel, "can_id": np.broadcast_to(can_id, rel.shape).astype(int),
                       "src": src, "label": label})
    pl = np.atleast_2d(pl)
    for k in range(8):
        df[f"b{k}"] = pl[:, k]
    return df

def build_legit(duration, drive, rng):
    """Release times + payloads of all legitimate frames (before bus arbitration)."""
    parts = []
    for row in can_matrix.itertuples():
        if np.isfinite(row.period_ms):
            T = row.period_ms / 1000
            T_real = T / (1 + ECU_SKEW_PPM[row.sender] * 1e-6)   # per-ECU clock skew
            phase = rng.uniform(0, T)                              # per-identifier phase
            k = np.arange(int((duration - phase) / T_real) + 1)
            rel = phase + k * T_real + rng.normal(0, row.jitter_ms / 1000, k.size)
            idx = np.clip(np.round(rel / drive["dt"]).astype(int), 0, drive["t"].size - 1)
            pl = encode(row.can_id, drive[SIGNALS[row.can_id][0]][idx], k)
        else:
            ev = np.sort(rng.uniform(0, duration, rng.poisson(row.event_rate_hz * duration)))
            rel = (ev[:, None] + np.array([0.0, 0.02, 0.04])).ravel()
            pl = np.zeros((rel.size, 8), np.uint8)
            pl[:, 0] = np.repeat(rng.integers(1, 16, ev.size), 3) if ev.size else pl[:, 0]
        parts.append(frames_df(rel, row.can_id, 0, 0, pl))
    return pd.concat(parts, ignore_index=True)

def simulate_bus(frames, bitrate=BITRATE):
    """Non-preemptive priority bus. When idle, the pending frame with the lowest identifier is sent; a
    newer instance of the same (identifier, source) overwrites an unsent older one. 't' = end-of-frame time."""
    rel = frames["rel"].to_numpy(); cid = frames["can_id"].to_numpy().tolist(); src = frames["src"].to_numpy().tolist()
    dur = can_frame_bits(8) / bitrate
    order = np.argsort(rel, kind="stable").tolist(); rel = rel.tolist()
    n = len(rel); t_eof = np.full(n, np.nan); dropped = np.zeros(n, bool)
    pending, heap, i, now = {}, [], 0, 0.0
    while i < n or heap:
        if not heap:
            now = max(now, rel[order[i]])
        while i < n and rel[order[i]] <= now:
            j = order[i]; key = (cid[j], src[j])
            if key in pending:
                dropped[pending[key]] = True
            pending[key] = j; heapq.heappush(heap, (cid[j], rel[j], j)); i += 1
        while heap and dropped[heap[0][2]]:
            heapq.heappop(heap)
        if not heap:
            continue
        _, _, j = heapq.heappop(heap)
        del pending[(cid[j], src[j])]
        now += dur; t_eof[j] = now
    out = frames.loc[~dropped].copy(); out["t"] = t_eof[~dropped]
    return out.sort_values("t", kind="stable").reset_index(drop=True)

def payload(df):
    return df[[f"b{k}" for k in range(8)]].to_numpy().astype(np.uint8)

drive_train = driving_profile(120, np.random.default_rng(seeds[0]))
train = simulate_bus(build_legit(120, drive_train, np.random.default_rng(seeds[1])))
load = len(train) * FRAME_TIME / train.t.max()
print(f"Training drive: {len(train):,} frames in {train.t.max():.0f} s "
      f"({len(train) / train.t.max():.0f} frames/s, worst-case bus load {load:.1%})")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
win = train[(train.t > 20) & (train.t < 20.25)]
ids_sorted = sorted(train.can_id.unique())
pos = {c: i for i, c in enumerate(ids_sorted)}
ax = axes[0]
ax.scatter(win.t - 20, win.can_id.map(pos), marker="|", s=120, color=OKABE_ITO[4])
ax.set_yticks(range(len(ids_sorted)), [f"0x{c:03X}" for c in ids_sorted])
ax.set_xlabel("Time [s] (250 ms excerpt)"); ax.set_title("Bus raster: each tick is one frame")

ax = axes[1]
d = train.groupby("can_id")["t"].diff()
for c, col in zip([0x0B0, 0x1A0, 0x2C0, 0x316], OKABE_ITO[1:5]):
    dd = (d[train.can_id == c].dropna() * 1e3).to_numpy()
    ax.hist(dd - np.median(dd), bins=np.linspace(-1.0, 1.0, 50), alpha=0.75, color=col,
            label=f"0x{c:03X} ({ID_NAME[c]}, T={np.median(dd):.2f} ms)")
ax.set_xlabel("Inter-arrival minus per-ID median [ms]"); ax.set_ylabel("Count")
ax.set_title("Per-ID inter-arrival: jitter + queuing"); ax.legend(fontsize=7)

ax = axes[2]
for c, col in [(0x0B0, OKABE_ITO[4]), (0x120, OKABE_ITO[5])]:
    m = (train.can_id == c).to_numpy()
    ax.plot(train.t.to_numpy()[m], decode(c, payload(train[train.can_id == c])), lw=0.8, color=col,
            label=f"{SIGNALS[c][0]} decoded from 0x{c:03X}")
ax.set_xlabel("Time [s]"); ax.set_ylabel("km/h  |  degrees"); ax.set_title("Signals decoded from the bus")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

pl = payload(train); cid = train.can_id.to_numpy()
prot = np.isin(cid, list(SIGNALS))
ok = all(np.array_equal(pl[cid == c, 7], checksum(pl[cid == c, :7], c)) for c in SIGNALS)
print(f"Checksums valid on all {prot.sum():,} protected training frames: {ok}")

## 4. Anomalies used to test the detectors

We now drive a 6-minute **test** journey and insert six anomaly episodes with known ground truth. Each episode models,
at a high level, a pattern an intrusion detector is expected to catch. The reference threat model is a node that has
gained write access to the chassis bus — for example through a compromised telematics/infotainment unit (the route taken
in the remote Jeep Cherokee study of Miller & Valasek, 2015) or a malicious OBD-II dongle.

| Episode | Pattern | Description | Property exploited |
|---|---|---|---|
| A | **DoS flooding** | identifier `0x000` emitted every 0.3 ms (the HCRL dataset's injection rate) | priority |
| B | **Fuzzing** | random identifiers and random payloads every 0.5 ms | no authentication |
| C | **Spoofing / injection** | forged `VehicleSpeed = 0 km/h` on `0x0B0` every 1 ms, with a valid checksum but the injector's own counter | no authentication |
| D | **Replay** | a recorded 5 s of `SteeringAngle` frames re-sent verbatim | broadcast + no freshness |
| E | **Suspension** | the gear ECU (`0x2C0`) falls silent for 10 s (e.g. forced into bus-off, Section 8) | error handling |
| F | **Masquerade** | the gear ECU is silent while another node continues `0x2C0` with the right period, a continued counter and a valid checksum, holding the gear value frozen | all three |

Suspension and masquerade correspond to the *weak* and *strong* adversary of Cho & Shin (2016); the bus-off mechanism
that can silence an ECU is examined in Section 8.

In [ ]:
EPISODES = pd.DataFrame([
    ("A: DoS 0x000",        60.0,  62.0),
    ("B: Fuzzing",         100.0, 103.0),
    ("C: Spoofing 0x0B0",  140.0, 145.0),
    ("D: Replay 0x120",    180.0, 185.0),
    ("E: Suspension 0x2C0", 220.0, 230.0),
    ("F: Masquerade 0x2C0", 260.0, 320.0),
], columns=["attack", "start", "end"]).set_index("attack")
TEST_DURATION = 360.0

def build_test(rng):
    drive = driving_profile(TEST_DURATION, rng)
    legit = build_legit(TEST_DURATION, drive, rng)
    extra = []
    s, e = EPISODES.loc["A: DoS 0x000"]
    rel = np.arange(s, e, 0.0003)
    extra.append(frames_df(rel, 0x000, 1, 1, np.zeros((rel.size, 8), np.uint8)))
    s, e = EPISODES.loc["B: Fuzzing"]
    rel = np.arange(s, e, 0.0005)
    extra.append(frames_df(rel, rng.integers(0, 0x800, rel.size), 1, 2, rng.integers(0, 256, (rel.size, 8)).astype(np.uint8)))
    s, e = EPISODES.loc["C: Spoofing 0x0B0"]
    rel = np.arange(s, e, 0.001)
    extra.append(frames_df(rel, 0x0B0, 1, 3, encode(0x0B0, np.zeros(rel.size), np.arange(rel.size))))
    s, e = EPISODES.loc["D: Replay 0x120"]
    rec = train[(train.can_id == 0x120) & (train.t >= 10) & (train.t < 15)]
    extra.append(frames_df(s + rec.t.to_numpy() - 10, 0x120, 1, 4, payload(rec)))
    for name in ["E: Suspension 0x2C0", "F: Masquerade 0x2C0"]:
        s, e = EPISODES.loc[name]
        legit = legit[~((legit.can_id == 0x2C0) & (legit.rel >= s) & (legit.rel < e))]
    s, e = EPISODES.loc["F: Masquerade 0x2C0"]
    T = 0.050 / (1 + ECU_SKEW_PPM["HU"] * 1e-6)      # attacker's own (head-unit) clock
    rel = s + np.arange(0, e - s, T)
    frozen_gear = np.full(rel.size, drive["gear"][int(s / drive["dt"])])
    extra.append(frames_df(rel, 0x2C0, 1, 6, encode(0x2C0, frozen_gear, np.arange(rel.size) + 7)))
    allf = pd.concat([legit] + extra, ignore_index=True)
    return simulate_bus(allf), drive

test, drive_test = build_test(np.random.default_rng(seeds[2]))
y = test.label.to_numpy()
print(f"Test drive: {len(test):,} frames, {(y > 0).sum():,} anomalous "
      f"({(y > 0).mean():.1%}).  Episode frame counts:")
print(test[test.label > 0].groupby("label").size().rename(index={1:"A DoS",2:"B Fuzz",3:"C Spoof",4:"D Replay",6:"F Masq"}).to_string())
print("(Episode E is a silence, so it adds no frames; it removes 0x2C0 frames.)")

## 5. Detector (a): timing / frequency, per identifier

The oldest and cheapest idea (Song et al., 2016): each periodic identifier arrives at a near-constant interval, so learn
the normal inter-arrival distribution **per identifier** on clean data and flag frames that arrive too soon.

For identifier $m$ with clean inter-arrival mean $\mu_m$ and standard deviation $\sigma_m$, flag a frame whose gap to the
previous frame of the same identifier satisfies $\Delta t < \mu_m - \kappa\,\sigma_m$. A flood or an injection on an
existing identifier *doubles the frame rate*, so the gap halves and the rule fires. The threshold $\kappa$ is the single
knob and is swept later for the ROC. Any identifier never seen in training is itself an anomaly (fuzzing, DoS on `0x000`).

In [ ]:
def learn_timing(clean):
    stats = {}
    for c, g in clean.groupby("can_id"):
        dt = np.diff(np.sort(g.t.to_numpy()))
        if dt.size >= 5 and np.isfinite(can_matrix.set_index('can_id').period_ms.get(c, np.nan)):
            stats[c] = (float(dt.mean()), float(dt.std() + 1e-6))
    return stats

def timing_score(frames, stats):
    """Per-frame anomaly score: how many sigmas EARLY the frame is (unknown ID -> large score). Higher = more anomalous."""
    t = frames.t.to_numpy(); cid = frames.can_id.to_numpy()
    score = np.zeros(len(frames)); last = {}
    for i in range(len(frames)):
        c = cid[i]
        if c not in stats:
            score[i] = 50.0; continue                 # identifier never seen on this bus
        mu, sd = stats[c]
        if c in last:
            gap = t[i] - last[c]
            score[i] = max(0.0, (mu - gap) / sd)      # only EARLY arrivals are suspicious
        last[c] = t[i]
    return score

timing_stats = learn_timing(train)
s_time = timing_score(test, timing_stats)
print(f"Learned timing model for {len(timing_stats)} periodic identifiers.")
print(f"Score at a known periodic ID vs during the DoS episode: "
      f"median(normal)={np.median(s_time[y==0]):.2f}, median(DoS)={np.median(s_time[y==1]):.2f}")

## 6. Detector (b): payload / semantic plausibility

Timing says nothing about *content*. A masquerade attack keeps the right period, so only the payload betrays it. We check
three physics- and protocol-based rules on the decoded signals, learned or derived from clean data:

1. **Counter continuity** — the 4-bit alive counter must advance by 1 (mod 16) between consecutive frames of a protected identifier. A second transmitter on the same identifier breaks the sequence.
2. **Checksum** — the E2E checksum must match. (A careful attacker recomputes it, so this alone is weak — which is the whole point of adding a MAC in Section 9.)
3. **Physical rate limit** — a signal cannot jump faster than the vehicle physically allows. We learn the 99.9th percentile of $|\Delta \text{signal}|/\Delta t$ on clean data and flag larger jumps (e.g. speed falling from 50 to 0 km/h in one frame).

In [ ]:
def learn_rate_limits(clean):
    lim = {}
    for c in SIGNALS:
        g = clean[clean.can_id == c].sort_values("t")
        if len(g) < 10:
            continue
        v = decode(c, payload(g)); dt = np.diff(g.t.to_numpy())
        rate = np.abs(np.diff(v)) / np.maximum(dt, 1e-4)
        lim[c] = float(np.quantile(rate, 0.999) * 1.5 + 1e-6)
    return lim

def payload_scores(frames, limits):
    """Return three boolean alarm vectors: counter break, checksum fail, physical-rate violation."""
    n = len(frames); cid = frames.can_id.to_numpy(); t = frames.t.to_numpy(); pl = payload(frames)
    counter_bad = np.zeros(n, bool); csum_bad = np.zeros(n, bool); rate_bad = np.zeros(n, bool)
    for c in SIGNALS:
        idx = np.flatnonzero(cid == c)
        if idx.size == 0:
            continue
        csum_bad[idx] = pl[idx, 7] != checksum(pl[idx, :7], c)
        # robust counter rule: a single monotonic sender advances +1 (and tolerates a few dropped
        # frames, +2..+8). A stall (delta 0, a duplicate) or a backward step (delta 9..15) means a
        # SECOND transmitter is interleaving its own counter -> injection or replay.
        delta = (pl[idx][1:, 6].astype(int) - pl[idx][:-1, 6].astype(int)) % 16
        counter_bad[idx[1:]] = (delta == 0) | (delta >= 9)
        if c in limits:
            v = decode(c, pl[idx]); dt = np.diff(t[idx])
            rate_bad[idx[1:]] = (np.abs(np.diff(v)) / np.maximum(dt, 1e-4)) > limits[c]
    return counter_bad, csum_bad, rate_bad

rate_limits = learn_rate_limits(train)
cnt_bad, cs_bad, rt_bad = payload_scores(test, rate_limits)
s_payload = (cnt_bad | cs_bad | rt_bad).astype(float)

# a frame is "in an episode" if its timestamp falls inside any attack window (so injected frames AND
# the genuine frames they collide with both count as inside the event, which is what matters for an alarm)
t_test = test.t.to_numpy()
in_episode = np.zeros(len(test), bool)
for _, (s, e) in EPISODES.iterrows():
    in_episode |= (t_test >= s) & (t_test < e)
for name, v in [("counter break", cnt_bad), ("checksum fail", cs_bad), ("rate violation", rt_bad)]:
    print(f"{name:16s}: {v.sum():5d} alarms, {(v & in_episode).sum():5d} inside an attack window, "
          f"{(v & ~in_episode).sum():4d} during benign driving")
print("\nInjection (C) and replay (D) break counter continuity and the physical rate limit; the")
print("masquerade (F) continues the counter seamlessly, so the payload rules barely see it -> Section 8.")

## 7. Detector (c): unsupervised ML on sliding windows

Timing and payload rules are hand-crafted. A complementary, model-light approach treats a **sliding window** of the bus
as a feature vector and learns, from clean data only, what "normal" looks like. Two classic devices:

* **Identifier entropy** (Müter & Asaj, 2011). Over a window of $W$ frames, the Shannon entropy of the identifier
  distribution $H = -\sum_m p_m \log_2 p_m$ is stable in normal traffic. A flood collapses it (one identifier dominates);
  fuzzing inflates it (many rare identifiers). This is a one-line statistic with no training beyond a clean mean/variance.
* **Isolation Forest** (Liu et al., 2008) on per-window features — frame rate, identifier entropy, number of distinct
  identifiers, fraction of unknown identifiers, mean/variance of payload bytes. It is unsupervised, fast on CPU, and
  gives a continuous anomaly score we can put on the same ROC as the others.

We assign each window's score back to the frames inside it, so all detectors are compared on the same per-frame labels.

In [ ]:
def window_features(frames, known_ids, W=200):
    """Non-overlapping windows of W frames -> feature matrix and the frame indices in each window."""
    cid = frames.can_id.to_numpy(); t = frames.t.to_numpy(); pl = payload(frames).astype(float)
    feats, groups = [], []
    for a in range(0, len(frames) - W + 1, W):
        sl = slice(a, a + W); w = cid[sl]
        _, counts = np.unique(w, return_counts=True); p = counts / counts.sum()
        H = float(-(p * np.log2(p)).sum())
        dur = max(t[a + W - 1] - t[a], 1e-6)
        unknown = np.mean(~np.isin(w, list(known_ids)))
        feats.append([W / dur, H, len(counts), unknown, pl[sl].mean(), pl[sl].var()])
        groups.append((a, a + W))
    cols = ["rate", "id_entropy", "n_ids", "frac_unknown", "byte_mean", "byte_var"]
    return pd.DataFrame(feats, columns=cols), groups

known_ids = set(can_matrix.can_id)
Ftr, _ = window_features(train, known_ids)
iforest = IsolationForest(n_estimators=200, contamination="auto", random_state=SEED).fit(Ftr)

Fte, groups = window_features(test, known_ids)
win_score = -iforest.score_samples(Fte)                  # higher = more anomalous
win_entropy = Fte.id_entropy.to_numpy()

# scatter back to frames
s_iforest = np.zeros(len(test)); s_entropy = np.zeros(len(test))
mu_H, sd_H = Ftr.id_entropy.mean(), Ftr.id_entropy.std()
for (a, b), si, H in zip(groups, win_score, win_entropy):
    s_iforest[a:b] = si
    s_entropy[a:b] = abs(H - mu_H) / (sd_H + 1e-9)        # deviation of window entropy from clean, in sigmas

fig, ax = plt.subplots(figsize=(11, 3.2))
tw = np.array([test.t.to_numpy()[a] for a, _ in groups])
ax.plot(tw, win_entropy, color=OKABE_ITO[4], lw=1.2, label="identifier entropy per window")
ax.axhline(mu_H, color=OKABE_ITO[2], ls="--", lw=1, label=f"clean mean = {mu_H:.2f} bits")
for name, (s, e) in EPISODES.iterrows():
    ax.axvspan(s, e, color=OKABE_ITO[5], alpha=0.12)
    ax.text((s + e) / 2, ax.get_ylim()[1], name.split(":")[0], ha="center", va="top", fontsize=8)
ax.set_xlabel("Time [s]"); ax.set_ylabel("Entropy [bits]")
ax.set_title("Identifier-distribution entropy over the drive (shaded = anomaly episodes)")
ax.legend(loc="lower right", fontsize=8); plt.tight_layout(); plt.show()
print(f"Entropy collapses during the DoS flood and rises during fuzzing, "
      f"as expected; quiet injection (C/F) barely moves it.")

## 8. The hard case: masquerade, clock-skew fingerprinting and the bus-off mechanism

The masquerade (episode F) defeats every detector so far: the period is right, the counter continues, the checksum is
valid and the gear value stays within physical limits. What *cannot* be forged easily is the **physical oscillator** of
the sending ECU. Every quartz clock has a small, stable frequency error (its *skew*, tens of ppm). Cho & Shin's CIDS
(2016) estimates each identifier's transmitter skew from the slow drift of its arrival times and raises an alarm when the
skew shifts — because a different ECU is now sending that identifier.

We estimate skew as the slope of the **accumulated clock offset** $O_k = t_k - t_0 - k\,T$ against message index $k$,
where $T$ is the learned mean period. On clean traffic the slope is a fixed value set by the genuine ECU's oscillator;
when the masquerading node (a different oscillator) takes over, the slope changes by the skew difference.

Sagong et al. (2018) showed this fingerprint can itself be *spoofed* by an attacker who deliberately emulates the
victim's skew — one more round in the detection/evasion contest.

The skew signal is in the *microsecond* drift of arrival times, far below the millisecond queuing jitter of a busy bus,
so CIDS is estimated on a lightly loaded or pre-filtered stream (or with a recursive least-squares identifier and a CUSUM
change detector, as in the original paper). We therefore demonstrate the principle on a **dedicated clean arrival stream**
for the gear identifier: the genuine TCU transmits for the first half, then a masquerading node with a different
oscillator takes over. The accumulated offset $O_k$ is a straight line whose slope is the skew; the slope changes at the
takeover even though period, counter and checksum are all correct.

In [ ]:
def arrival_stream(period, skew_ppm, n, t0, jitter_s, rng):
    """Clean periodic arrivals of one ECU: true period = nominal / (1 + skew)."""
    step = period / (1 + skew_ppm * 1e-6)
    return t0 + np.cumsum(np.full(n, step)) + rng.normal(0, jitter_s, n)

rng_sk = np.random.default_rng(seeds[3])
T0 = 0.050                                                # nominal gear period
n_half = 1000
t_gen = arrival_stream(T0, ECU_SKEW_PPM["TCU"], n_half, 0.0, 20e-6, rng_sk)    # genuine TCU, -45 ppm
t_atk = arrival_stream(T0, ECU_SKEW_PPM["HU"], n_half, t_gen[-1], 20e-6, rng_sk)  # masquerade, -80 ppm
arr = np.concatenate([t_gen, t_atk]); k = np.arange(arr.size)
O = (arr - arr[0] - k * T0) * 1e3                         # accumulated offset [ms]

def win_skew(arr, T, window=150, step=25):
    cen, sk = [], []
    kk = np.arange(window)
    for a in range(0, arr.size - window, step):
        sk.append((np.polyfit(kk, arr[a:a + window], 1)[0] / T - 1) * 1e6)
        cen.append(arr[a + window // 2])
    return np.array(cen), np.array(sk)

tc, sk = win_skew(arr, T0)
switch_t = t_gen[-1]
clean = tc < switch_t
mu_sk, sd_sk = sk[clean].mean(), sk[clean].std()

fig, axes = plt.subplots(1, 2, figsize=(13, 3.4))
ax = axes[0]
ax.plot(arr, O, color=OKABE_ITO[4], lw=1.0)
ax.axvline(switch_t, color=OKABE_ITO[5], ls="--", label="masquerade takes over")
ax.set_xlabel("Time [s]"); ax.set_ylabel("Accumulated offset [ms]")
ax.set_title("Accumulated clock offset of 0x2C0 (slope = skew)"); ax.legend(fontsize=8)
ax = axes[1]
ax.plot(tc, sk, color=OKABE_ITO[4], lw=1.5, label="windowed skew estimate")
ax.axhspan(mu_sk - 4 * sd_sk, mu_sk + 4 * sd_sk, color=OKABE_ITO[2], alpha=0.18,
           label=f"genuine TCU band ({mu_sk:.0f} ± {4*sd_sk:.0f} ppm)")
ax.axvline(switch_t, color=OKABE_ITO[5], ls="--")
flag = np.abs(sk - mu_sk) > 4 * sd_sk
ax.scatter(tc[flag], sk[flag], color=OKABE_ITO[5], s=14, zorder=5, label="change detected")
ax.set_xlabel("Time [s]"); ax.set_ylabel("Clock skew [ppm]")
ax.set_title("Skew fingerprint separates the two ECUs"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

print(f"Genuine TCU skew {mu_sk:+.0f} ppm (sd {sd_sk:.1f}); after takeover {sk[~clean].mean():+.0f} ppm "
      f"-> a {sk[~clean].mean() - mu_sk:+.0f} ppm step, matching the "
      f"{abs(ECU_SKEW_PPM['HU'] - ECU_SKEW_PPM['TCU']):.0f} ppm difference between the two oscillators.")
print("Period, counter and checksum are all correct here; only the physical skew betrays the masquerade.")

### How an ECU is silenced: the bus-off attack

Episodes E and F assume the genuine gear ECU can be made to fall silent. Cho & Shin (2016) describe how: the CAN error
handling that is meant to improve reliability can be turned against a node. Each controller keeps a Transmit Error
Counter (TEC). An attacker that transmits a dominant bit exactly over a victim's recessive bit — at the same moment the
victim transmits — causes a bit error; the victim increments its TEC by 8 and re-transmits, hitting the same crafted
collision again. After the TEC passes 255 the controller enters **bus-off** and stops transmitting. The model below just
illustrates the counter dynamics; it is not an attack tool.

In [ ]:
def tec_trajectory(n_frames, attacked):
    """Illustrative CAN TEC dynamics: +8 on a transmit error, -1 on a successful transmit; bus-off at 256."""
    tec = 0; traj = []
    for i in range(n_frames):
        tec += 8 if attacked else -1
        tec = max(tec, 0)
        traj.append(tec)
        if tec >= 256:                                   # error-passive at 128, bus-off at 256
            traj += [256] * (n_frames - i - 1)
            break
    return np.array(traj)

fig, ax = plt.subplots(figsize=(7.5, 3.0))
ax.plot(tec_trajectory(60, True), color=OKABE_ITO[5], lw=1.8, label="under a targeted bit-error attack")
ax.plot(tec_trajectory(60, False) + 40, color=OKABE_ITO[1], lw=1.8, ls="--", label="normal operation (decreasing)")
ax.axhline(128, color=OKABE_ITO[0], ls=":", label="error-passive (TEC = 128)")
ax.axhline(256, color="black", ls=":", label="bus-off (TEC = 256)")
ax.set_xlabel("Transmitted frames"); ax.set_ylabel("Transmit Error Counter")
ax.set_title("Why error handling makes CAN vulnerable (Cho & Shin, 2016)")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()
print("Roughly 32 forced errors (32 x 8 = 256) drive the victim into bus-off and silence it.")

## 9. Evaluation: precision, recall, F1, ROC, latency and false alarms per hour

A detector that catches everything but fires constantly is useless in a vehicle. We report, per detector:

* **Precision, recall, F1** against the per-frame ground truth at a chosen operating point.
* **ROC and AUC** by sweeping the score threshold of the continuous detectors — threshold-independent.
* **Detection latency** — time from an episode's first anomalous frame to the first alarm inside it.
* **False-alarm rate during benign driving** — measured only on frames *outside* every attack window, and counted as debounced **alarm events** (a fresh event requires a quiet gap), because that is what a security-operations team actually lives with. Extrapolated to alarms per hour.

We evaluate the four on-bus detectors. The clock-skew fingerprint of Section 8 needs a lightly loaded or pre-filtered
stream, so it is kept as the conceptual answer to the masquerade rather than scored on the busy bus. No single detector
wins on every episode, which is the central lesson.

In [ ]:
benign = ~in_episode                                       # frames outside every attack window

detectors = {
    "Timing (early arrival)": s_time,
    "Payload (counter/checksum/rate)": s_payload,
    "ID entropy (window)": s_entropy,
    "Isolation Forest (window)": s_iforest,
}
op_thr = {"Timing (early arrival)": 4.0, "Payload (counter/checksum/rate)": 0.5,
          "ID entropy (window)": 4.0,
          "Isolation Forest (window)": float(np.quantile(-iforest.score_samples(Ftr), 0.999)),
          "Clock skew (0x2C0)": 4.0}

def alarm_events_per_hour(alarm, frames, benign, gap_s=0.5):
    """Count debounced alarm EVENTS during benign driving, per hour. A new event needs a quiet gap."""
    t = frames.t.to_numpy()
    idx = np.flatnonzero(alarm & benign)
    if idx.size == 0:
        return 0.0
    ta = np.sort(t[idx])
    events = 1 + int(np.sum(np.diff(ta) > gap_s))
    dur_h = (t[benign].max() - t[benign].min()) / 3600
    return events / max(dur_h, 1e-9)

def episode_recall_latency(alarm, frames, y):
    t = frames.t.to_numpy(); rec, lat = {}, {}
    for name, (s, e) in EPISODES.iterrows():
        in_ep = (t >= s) & (t < e)
        anom = in_ep & (y > 0)
        if name.startswith("E"):                         # suspension: success = NO alarm storm on 0x2C0 absence
            continue
        caught = alarm & anom
        rec[name] = caught.sum() / max(anom.sum(), 1)
        fired = np.flatnonzero(caught)
        lat[name] = (t[fired[0]] - t[np.flatnonzero(anom)[0]]) if fired.size else np.nan
    return rec, lat

rows = []
for name, sc in detectors.items():
    al = sc >= op_thr[name]
    p, r, f, _ = precision_recall_fscore_support(y > 0, al, average="binary", zero_division=0)
    auc = roc_auc_score(y > 0, sc) if len(np.unique(sc)) > 2 else np.nan
    fa = alarm_events_per_hour(al, test, benign)
    rows.append((name, p, r, f, auc, fa))
summary = pd.DataFrame(rows, columns=["detector", "precision", "recall", "F1", "ROC_AUC", "benign_alarms_per_h"])
print(summary.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.3))
ax = axes[0]
for (name, sc), col in zip(detectors.items(), OKABE_ITO):
    if len(np.unique(sc)) < 3:
        continue
    fpr, tpr, _ = roc_curve(y > 0, sc)
    ax.plot(fpr, tpr, color=col, lw=1.8, label=f"{name.split(' (')[0]} (AUC {roc_auc_score(y>0, sc):.3f})")
ax.plot([0, 1], [0, 1], "k--", lw=0.8)
ax.set_xlabel("False positive rate (per frame)"); ax.set_ylabel("True positive rate")
ax.set_title("ROC per detector"); ax.legend(fontsize=8, loc="lower right")

rec_tbl, lat_tbl = {}, {}
for name, sc in detectors.items():
    r, l = episode_recall_latency(sc >= op_thr[name], test, y)
    rec_tbl[name] = r
recdf = pd.DataFrame(rec_tbl)
ax = axes[1]
im = ax.imshow(recdf.to_numpy(), aspect="auto", cmap="viridis", vmin=0, vmax=1)
ax.set_xticks(range(recdf.shape[1]), [c.split(" (")[0] for c in recdf.columns], rotation=25, ha="right", fontsize=8)
ax.set_yticks(range(recdf.shape[0]), [i.split(":")[0] for i in recdf.index], fontsize=9)
for (r, c), v in np.ndenumerate(recdf.to_numpy()):
    ax.text(c, r, f"{v:.2f}", ha="center", va="center", color="white" if v < 0.5 else "black", fontsize=8)
ax.set_title("Per-episode recall (fraction of anomalous frames caught)")
fig.colorbar(im, ax=ax, label="recall")
plt.tight_layout(); plt.show()

best = summary.loc[summary.F1.idxmax()]
print(f"Highest single-detector F1: {best.detector} (F1={best.F1:.3f}, "
      f"{best.benign_alarms_per_h:.0f} benign alarms/h).")
f_recall = recdf.loc["F: Masquerade 0x2C0"].max()
print(f"The masquerade (F) is the hard case: best on-bus recall only {f_recall:.2f} -> it needs the "
      f"clock-skew fingerprint (Section 8) or, better, cryptographic authentication (Section 10).")

### Combining detectors, and the evasion problem

Because each detector covers different episodes, a **union** (fire if any detector fires) raises recall at the cost of
more benign alarms — the classic trade-off. An informed adversary, however, studies these same detectors:

* **Against timing**, inject only in place of a suppressed frame (the masquerade), so the rate never changes.
* **Against the checksum**, recompute it — trivial once the algorithm is known.
* **Against the counter**, continue the victim's sequence (our masquerade does exactly this).
* **Against clock-skew fingerprinting**, emulate the victim's skew (Sagong et al., 2018).

This is why detection alone is not a safety argument: it must be paired with **prevention** (cryptographic
authentication) and an assumption-aware TARA that records what each detector can and cannot see.

In [ ]:
union = np.zeros(len(test), bool)
for name, sc in detectors.items():
    union |= sc >= op_thr[name]
p, r, f, _ = precision_recall_fscore_support(y > 0, union, average="binary", zero_division=0)
fa = alarm_events_per_hour(union, test, benign)
print(f"Union of the four on-bus detectors: precision={p:.3f}, recall={r:.3f}, F1={f:.3f}, {fa:.0f} benign alarms/h")
rec_u, _ = episode_recall_latency(union, test, y)
print("Per-episode recall (union):", {k.split(':')[0]: f"{v:.2f}" for k, v in rec_u.items()})

## 10. Prevention: AUTOSAR SecOC and the cost of authentication

Detection observes; **prevention** stops a forged frame from being trusted. AUTOSAR *Secure Onboard Communication*
(SecOC) appends to each protected frame a **Message Authentication Code (MAC)** over the payload plus a **freshness
value** (a monotonic counter or time), using a symmetric key shared only by the genuine sender and the receivers
(AUTOSAR CP R24-11, document 654). A masquerading ECU without the key cannot produce a valid MAC, and the freshness value
makes a replayed frame stale — closing exactly the gaps (F and D) that detection struggled with.

The catch is **space**. A classic CAN frame has only 8 data bytes. SecOC therefore allows **truncation**: *SecOC Profile 1
(24Bit-CMAC-8Bit-FV)* uses an AES-128 CMAC truncated to the 24 most significant bits and an 8-bit truncated freshness
value — 4 bytes of overhead, leaving 4 bytes for the signal. NIST (SP 800-38B appendix) notes that a truncated MAC of
$t$ bits leaves a forgery probability of about $2^{-t}$ per attempt; 24 bits is a deliberate trade-off for a bus where
an attacker also faces arbitration and error limits. More MAC bits mean either fewer signal bytes or a second frame.

In [ ]:
SECOC_PROFILES = {
    "Profile 1 (24-bit CMAC + 8-bit FV)": (3, 1),
    "Profile 3 / JASPAR (24-bit CMAC, FV elsewhere)": (3, 0),
    "Stronger (64-bit MAC + 32-bit FV, two frames)": (8, 4),
}

def secoc_overhead(mac_bytes, fv_bytes, payload_bytes=8, fd=False):
    """Frames needed and bus-time ratio when adding (MAC+FV) bytes to a payload."""
    total = payload_bytes + mac_bytes + fv_bytes
    if not fd:
        frames = int(np.ceil(total / 8))
        base = can_frame_bits(min(payload_bytes, 8))
        auth = sum(can_frame_bits(min(8, total - 8 * i)) for i in range(frames))
    else:
        frames = 1                                     # CAN FD carries up to 64 data bytes in one frame
        base = canfd_frame_bits(payload_bytes)
        auth = canfd_frame_bits(total)
    return frames, auth / base

def canfd_frame_bits(data_bytes):
    """Approximate CAN FD frame length (11-bit ID, BRS): ~29-bit arbitration phase at nominal rate +
    data phase (header ~28 bits + payload + 21-bit CRC/stuff) at the data rate; here reported in bit-times."""
    return 29 + 28 + 8 * data_bytes + 27            # simplified, enough for a relative comparison

rows = []
for name, (mac, fv) in SECOC_PROFILES.items():
    f_c, r_c = secoc_overhead(mac, fv, 8, fd=False)
    f_f, r_f = secoc_overhead(mac, fv, 8, fd=True)
    rows.append((name, mac + fv, f_c, f"{(r_c - 1) * 100:+.0f}%", f"{(r_f - 1) * 100:+.0f}%"))
over = pd.DataFrame(rows, columns=["SecOC configuration", "overhead_bytes", "classic_CAN_frames",
                                   "classic_bus_time", "CAN_FD_bus_time"])
print(over.to_string(index=False))
print("\nClassic CAN often needs a SECOND frame once 8 payload bytes are already used, roughly doubling")
print("the bus time for that message; CAN FD (up to 64 data bytes) absorbs the MAC in one frame.")

In [ ]:
# bus-load impact of authenticating the four fastest periodic IDs on our bus
fast = PERIODIC_RATE = can_matrix.dropna(subset=["period_ms"]).copy()
fast["rate_hz"] = 1000.0 / fast["period_ms"]
base_load = (fast["rate_hz"] * can_frame_bits(8)).sum() / BITRATE
# Profile 1: +4 bytes -> 12 bytes -> 2 classic frames for each protected ID
prot_ids = list(SIGNALS)
auth_bits = fast.apply(lambda r: (2 * can_frame_bits(8) if r["can_id"] in prot_ids else can_frame_bits(8)), axis=1)
auth_load = (fast["rate_hz"] * auth_bits).sum() / BITRATE

fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.bar(["Unauthenticated", "SecOC Profile 1\n(classic CAN, 2 frames)"], [base_load * 100, auth_load * 100],
       color=[OKABE_ITO[1], OKABE_ITO[5]])
for i, v in enumerate([base_load, auth_load]):
    ax.text(i, v * 100 + 0.5, f"{v:.1%}", ha="center")
ax.set_ylabel("Worst-case bus load [%]")
ax.set_title("Bus load before and after authenticating the periodic signals (500 kbit/s)")
plt.tight_layout(); plt.show()
print(f"Authenticating the {len(prot_ids)} signal IDs raises worst-case bus load from "
      f"{base_load:.1%} to {auth_load:.1%} on classic CAN -> a strong argument for CAN FD on new platforms.")

Beyond per-frame authentication, defence-in-depth adds **domain gateways and segmentation** (the central gateway in the
Jeep exploit became the pivot point; segmenting the infotainment domain from the powertrain bus and filtering what may
cross is now standard), **hardware security modules** for key storage, and **secure boot / secure diagnostics** so that
the re-flashing step of that attack is itself blocked.

Looking ahead, **CAN XL** (ISO 11898-1:2024, 3rd edition) raises the payload to 2048 bytes and splits the data link layer
into LLC and MAC sublayers; its security add-on **CANsec** (CiA 613-2) reuses MACsec-style integrity, freshness,
authenticity and optional confidentiality. As of October 2026 CiA 613-2 v1.0.0 is at *draft specification proposal* stage
and member-restricted, so CANsec is best described as emerging rather than deployed.

## 11. Putting it in the standards: ISO/SAE 21434 TARA and UN R155

A vehicle cybersecurity case does not list detectors; it argues about **risk**. ISO/SAE 21434 structures this as a TARA:
for each asset and damage scenario, rate the **impact** and the **attack feasibility**, combine them into a risk value,
and decide on treatment. The attack-feasibility bands and the risk matrix below are exactly those implemented in the
repository's `avsafety.iso21434` module, so the numbers here match the toolkit.

UN Regulation No. 155 makes a cybersecurity management system mandatory for type approval and gives, in **Annex 5
Table A1**, a reference threat list. The communication-channel threats (section 4.3.2) map directly onto the episodes of
this notebook.

In [ ]:
# ISO/SAE 21434 attack-feasibility bands (Annex G, from ISO/IEC 18045) and risk matrix (Annex H) -- matches avsafety
def feasibility_from_potential(potential):
    return "High" if potential <= 13 else "Medium" if potential <= 19 else "Low" if potential <= 24 else "Very Low"

RISK_MATRIX = {
    "Severe":     {"Very Low": 2, "Low": 3, "Medium": 4, "High": 5},
    "Major":      {"Very Low": 1, "Low": 2, "Medium": 3, "High": 4},
    "Moderate":   {"Very Low": 1, "Low": 2, "Medium": 2, "High": 3},
    "Negligible": {"Very Low": 1, "Low": 1, "Medium": 1, "High": 1},
}

# TARA rows for the notebook's episodes. Attack potential is the sum of five factors (Annex G / ISO 18045);
# here we give a representative total and the resulting feasibility band.
tara = pd.DataFrame([
    # episode, R155 Annex 5 item, impact (safety), attack potential, detectable by, prevented by
    ("DoS 0x000 flood",  "4.3.2 #8.1 (DoS via communication)", "Major",   10, "timing, entropy", "rate limiting, gateway"),
    ("Fuzzing",          "4.3.2 #6.1 (untrusted messages)",    "Major",   10, "entropy, IForest", "SecOC (invalid MAC)"),
    ("Spoofing 0x0B0",   "4.3.2 #4.1 (spoofing by impers.)",   "Severe",  17, "timing, payload", "SecOC MAC"),
    ("Replay 0x120",     "4.3.2 #6.3 (replay)",                "Severe",  14, "counter, payload", "SecOC freshness value"),
    ("Masquerade 0x2C0", "4.3.2 #4.1 + bus-off (error handl.)","Severe",  20, "clock skew only", "SecOC MAC + freshness"),
], columns=["episode", "UN_R155_Annex5", "impact", "attack_potential", "detectable_by", "prevented_by"])
tara["feasibility"] = tara.attack_potential.map(feasibility_from_potential)
tara["risk_1to5"] = [RISK_MATRIX[i][f] for i, f in zip(tara.impact, tara.feasibility)]
print(tara.to_string(index=False))
print("\nThe same bands/matrix are in avsafety.iso21434 (feasibility_from_potential, risk_value).")

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 3.6))
impacts = ["Negligible", "Moderate", "Major", "Severe"]; feas = ["Very Low", "Low", "Medium", "High"]
M = np.array([[RISK_MATRIX[i][f] for f in feas] for i in impacts])
im = ax.imshow(M, cmap="YlOrRd", vmin=1, vmax=5, aspect="auto")
ax.set_xticks(range(4), feas); ax.set_yticks(range(4), impacts)
ax.set_xlabel("Attack feasibility"); ax.set_ylabel("Impact")
for (r, c), v in np.ndenumerate(M):
    ax.text(c, r, v, ha="center", va="center", fontweight="bold")
for _, row in tara.iterrows():
    ax.scatter(feas.index(row.feasibility), impacts.index(row.impact), s=260,
               facecolors="none", edgecolors=OKABE_ITO[4], linewidths=2)
    ax.annotate(row.episode.split()[0], (feas.index(row.feasibility), impacts.index(row.impact)),
                fontsize=7, ha="center", va="bottom", xytext=(0, 10), textcoords="offset points")
ax.set_title("ISO/SAE 21434 risk matrix (Annex H) with the notebook's threat scenarios")
fig.colorbar(im, ax=ax, label="risk value (1-5)")
plt.tight_layout(); plt.show()
print("The scenarios monitoring cannot prevent (spoofing, replay, masquerade) are exactly those that need")
print("authentication -- the TARA conclusion drives the SecOC requirement, closing the loop.")

## 12. Public datasets (for further study, no download needed)

To take this beyond synthetic traffic, two open datasets are widely used and do **not** need to be downloaded to follow
this notebook:

* **HCRL Car-Hacking dataset** (Seo et al., 2018; Song et al., 2020) — real CAN logged from a vehicle via the OBD-II
  port, with DoS, fuzzing and RPM/gear spoofing captures. Its injection rates (0.3 ms for DoS, 1 ms for spoofing) are the
  ones reproduced above.
* **ROAD dataset** (Verma et al., 2024, PLOS ONE) — a more recent set that includes *verified* fabrication and
  masquerade attacks and, importantly, a critical review of the flaws in earlier datasets (idle captures, no masquerade,
  trivially separable attacks). It is the recommended benchmark for honest IDS evaluation.

A good exercise is to retrain the detectors of Sections 5–7 on one of these and recompute the Section 9 metrics; expect
the masquerade and quiet-injection cases to be far harder than the floods, exactly as here.

## Key takeaways

1. CAN's three design properties — broadcast, priority by identifier, and no sender authentication — are the root cause of every bus attack; the identifier is a *priority*, not an address.
2. A faithful traffic generator needs **per-ECU clocks with per-identifier phase**, jitter and an arbitration-based bus model; a single global clock makes timing detection look better than it is.
3. Floods and fuzzing are easy (timing and identifier entropy catch them instantly); **spoofing and replay** need payload/counter/freshness checks; the **masquerade** defeats all on-bus heuristics and is why clock-skew fingerprinting and, decisively, cryptographic authentication exist.
4. Report detectors the way a programme must: precision/recall/F1 **and** detection latency **and** false alarms per hour of benign driving — a high recall with hundreds of benign alarms per hour is not shippable.
5. Every detector can be evaded by an adversary who models it; **AUTOSAR SecOC** (truncated MAC + freshness value) prevents forgery and replay, at a bus-load cost that is modest on **CAN FD** but can double frame counts on classic CAN.
6. The engineering decision is driven by an **ISO/SAE 21434 TARA** and the **UN R155 Annex 5** threat list: the highest-risk scenarios are exactly those requiring authentication rather than monitoring alone.

## Self-assessment

1. Why does an attacker choose identifier `0x000` for a denial-of-service flood rather than a high identifier?
2. A masquerade attack keeps the correct period, counter and checksum. Which of the four on-bus detectors, if any, can still catch it, and which physical property does the clock-skew method exploit?
3. Our timing detector raised over a thousand benign alarms per hour. Name two causes in the traffic model and one way to reduce them without missing a flood.
4. SecOC Profile 1 truncates the MAC to 24 bits. What is the approximate per-attempt forgery probability, and why might 24 bits nonetheless be acceptable on a CAN bus?
5. Place the replay attack in the ISO/SAE 21434 risk matrix used here (impact and feasibility) and state which single SecOC element defeats it.

<details>
<summary>Answers</summary>

1. The bus is a wired-AND where a dominant 0 wins arbitration, so the **lowest** identifier has the highest priority. `0x000` is all-dominant and wins against every other frame, letting a steady stream of it starve all legitimate traffic.
2. None of the four on-bus detectors catch it reliably (the notebook measures ~0 recall): period, counter and checksum are all correct and the signal stays within physical limits. **Clock-skew fingerprinting** (CIDS) can, because it measures the slow drift of arrival times set by the sending ECU's **physical oscillator**, which the masquerading node cannot match without deliberately emulating it.
3. Causes: **queuing delay** from arbitration (frames bunch up behind higher-priority traffic, so some gaps are very short) and **jitter** from task scheduling. Mitigations: raise the threshold $\kappa$, or require an early arrival to persist over several consecutive frames (debouncing) before alarming — floods produce *sustained* early arrivals, so they are still caught.
4. About $2^{-24} \approx 6\times10^{-8}$ per forgery attempt. It is acceptable because an on-bus attacker cannot make unlimited silent attempts: each wrong MAC is a detectable event, the bus error-handling and arbitration limit throughput, and the freshness value must also match — so the effective forgery rate is far lower than a pure $2^{-24}$ guessing budget on an offline oracle.
5. Impact **Severe** (a forged steering/speed value is safety-relevant) and feasibility about **Medium** (attack potential ≈ 14) give a risk value of **4** in the Annex H matrix used here. The **freshness value** (a monotonic counter or timestamp inside the authenticated data) defeats replay by making a re-sent old frame stale.
</details>

## References

Verified via Crossref, arXiv, USENIX, UNECE/EUR-Lex and the official standards bodies on 28 October 2026.

1. ISO 11898-1:2024, *Road vehicles — Controller area network (CAN) — Part 1: Data link layer and physical coding sublayer* (3rd ed., including CAN FD and CAN XL frame formats). https://www.iso.org/standard/86384.html
2. ISO/SAE 21434:2021, *Road vehicles — Cybersecurity engineering.*
3. UNECE, *UN Regulation No. 155 — Uniform provisions concerning the approval of vehicles with regards to cyber security and cyber security management system*, Annex 5 (threat list). Official Journal of the EU, 2021. https://eur-lex.europa.eu/eli/reg/2021/387/oj
4. AUTOSAR, *Specification of Secure Onboard Communication*, Classic Platform R24-11, document 654 (SecOC profiles, truncated MAC and freshness value). https://www.autosar.org
5. C. Miller and C. Valasek, *Remote Exploitation of an Unaltered Passenger Vehicle*, IOActive white paper, 5 August 2015 (Black Hat USA 2015 / DEF CON 23). https://ioactive.com/pdfs/IOActive_Remote_Car_Hacking.pdf
6. K.-T. Cho and K. G. Shin, *Error Handling of In-vehicle Networks Makes Them Vulnerable*, ACM CCS 2016. https://doi.org/10.1145/2976749.2978302
7. K.-T. Cho and K. G. Shin, *Fingerprinting Electronic Control Units for Vehicle Intrusion Detection*, USENIX Security 2016. https://www.usenix.org/conference/usenixsecurity16/technical-sessions/presentation/cho
8. H. M. Song, H. R. Kim and H. K. Kim, *Intrusion detection system based on the analysis of time intervals of CAN messages for in-vehicle network*, ICOIN 2016. https://doi.org/10.1109/ICOIN.2016.7427089
9. M. Müter and N. Asaj, *Entropy-based anomaly detection for in-vehicle networks*, IEEE IV 2011. https://doi.org/10.1109/IVS.2011.5940552
10. F. T. Liu, K. M. Ting and Z.-H. Zhou, *Isolation Forest*, IEEE ICDM 2008. https://doi.org/10.1109/ICDM.2008.17
11. E. Seo, H. M. Song and H. K. Kim, *GIDS: GAN based Intrusion Detection System for In-Vehicle Network*, PST 2018 (corrected version: arXiv:1907.07377). https://doi.org/10.1109/PST.2018.8514157
12. H. M. Song, J. Woo and H. K. Kim, *In-vehicle network intrusion detection using deep convolutional neural network*, Vehicular Communications 21, 2020 (HCRL Car-Hacking dataset). https://doi.org/10.1016/j.vehcom.2019.100198
13. M. E. Verma, R. A. Bridges, M. D. Iannacone, S. C. Hollifield, P. Moriano et al., *A comprehensive guide to CAN IDS data and introduction of the ROAD dataset*, PLOS ONE 19(1), 2024. https://doi.org/10.1371/journal.pone.0296879
14. S. U. Sagong, X. Ying, A. Clark, L. Bushnell and R. Poovendran, *Cloaking the Clock: Emulating Clock Skew in Controller Area Networks*, ACM/IEEE ICCPS 2018. https://doi.org/10.1109/ICCPS.2018.00012
15. R. I. Davis, A. Burns, R. J. Bril and J. J. Lukkien, *Controller Area Network (CAN) schedulability analysis: Refuted, revisited and revised*, Real-Time Systems 35(3), 2007. https://doi.org/10.1007/s11241-007-9012-7
16. CiA 613-2, *CAN XL add-on services — Part 2: CANsec data plane* (draft specification proposal v1.0.0, 2026; member-restricted). https://www.can-cia.org/can-knowledge/cia-613-series-can-xl-add-on-services

---

**Navigation:** ← [Attack surface analysis](03_attack_surface_analysis.ipynb) · [Module index](README.md) · [next module: AI Safety (ISO/PAS 8800) →](../06_AI_Safety_ISO_PAS_8800/01_ai_safety_standards.ipynb)

*Further practice:* the author's companion repository [AI-Perception-Safety-Workshop](https://github.com/milinpatel07/AI-Perception-Safety-Workshop).